# Surface Mapping Workshop — Scene Camera to Screen Coordinates

This notebook focuses on the core screen-mapping path: estimate AprilTag homographies, assign gaze samples to scene frames, transform gaze into screen pixels, and export the mapped data for the AOI workshop.

You will complete three helper functions, test each one on toy data, then run them on one Neon recording.


## Learning Goals

By the end, you should be able to:

1. Apply a homography to map a 2-D point between coordinate systems.
2. Assign gaze timestamps to the most recent scene-video frame.
3. Build a screen-space gaze heatmap.
4. Interpret mapping quality from AprilTag coverage and on-screen gaze percentage.


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nb_setup import DEFAULT_PARTICIPANT_ID, ensure_workshop_data, setup
PROJECT_ROOT = setup("screen_mapper.py")
ensure_workshop_data(PROJECT_ROOT)

from libs.analysis.screen_mapper import ScreenMapper, ScreenMapperConfig, compute_marker_margin_px
from libs.analysis.gaze_cleaning import clean_gaze_stream
from libs.analysis.recording_helpers import find_neon_recording, neon_participant_id
from libs.project_config import PLOT_COLORS


## Exercise 5 — Apply a Homography

A homography is a 3x3 matrix that maps scene-camera pixels into screen pixels. Convert `(x, y)` to homogeneous coordinates `[x, y, 1]`, multiply by `H`, then divide by the third coordinate.


In [ ]:
def apply_homography(H: np.ndarray, point: np.ndarray) -> np.ndarray:
    """Transform a 2-D point with a 3x3 homography matrix."""
    # Starter scaffold — fill in the ... blanks, then uncomment:
    # p = np.array([..., ..., 1.0])   # homogeneous coordinates
    # q = H @ ...
    # if np.isclose(q[2], 0.0):
    #     return np.array([np.nan, np.nan])
    # return np.array([q[0] / ..., q[1] / ...])
    raise NotImplementedError("Complete apply_homography")

In [ ]:
def test_apply_homography(fn):
    pt = np.array([100.0, 200.0])

    out_id = fn(np.eye(3), pt)
    assert isinstance(out_id, np.ndarray), "Return a numpy array."
    assert out_id.shape == (2,), "Output must have shape (2,)."
    assert np.allclose(out_id, pt), f"Identity H must return {pt}, got {out_id}."

    H_scale = np.diag([2.0, 3.0, 1.0])
    out_sc = fn(H_scale, pt)
    assert np.allclose(out_sc, [200.0, 600.0]), f"Expected [200, 600], got {out_sc}."

    print("Exercise 5 passed.")


test_apply_homography(apply_homography)


## Exercise 6 — Assign Gaze Timestamps to Scene Frames

Gaze is faster than the scene video. For each gaze timestamp, find the most recent frame timestamp at or before it. `np.searchsorted(..., side="right") - 1` does the lookup; clipping handles samples just outside the video timeline.


In [ ]:
def assign_gaze_to_frames(
    gaze_ts_ns: np.ndarray,
    scene_times_ns: np.ndarray,
) -> np.ndarray:
    """Return the nearest preceding scene-frame index for each gaze timestamp."""
    # Starter scaffold — fill in the ... blanks, then uncomment:
    # indices = np.searchsorted(..., ..., side="right") - 1
    # return np.clip(indices, 0, ... - 1).astype(int)
    raise NotImplementedError("Complete assign_gaze_to_frames")

In [ ]:
def test_assign_gaze_to_frames(fn):
    scene_ns = np.array([0, 33_000_000, 66_000_000, 100_000_000], dtype=np.int64)
    gaze_ns  = np.array([10_000_000, 40_000_000, 75_000_000, 100_000_000], dtype=np.int64)
    expected = np.array([0, 1, 2, 3])

    out = fn(gaze_ns, scene_ns)
    assert isinstance(out, np.ndarray), "Return a numpy array."
    assert np.array_equal(out, expected), f"Expected {expected}, got {out}."

    assert fn(np.array([-1_000_000], dtype=np.int64), scene_ns)[0] == 0
    assert fn(np.array([200_000_000], dtype=np.int64), scene_ns)[0] == len(scene_ns) - 1

    print("Exercise 6 passed.")


test_assign_gaze_to_frames(assign_gaze_to_frames)


## Exercise 7 — Build a Gaze Heatmap

A heatmap is a 2-D histogram of on-screen gaze positions. Use `np.histogram2d` with a fixed range of `[[0, screen_w], [0, screen_h]]` so different recordings stay comparable.


In [ ]:
def build_gaze_heatmap(
    xs: np.ndarray,
    ys: np.ndarray,
    screen_w: int,
    screen_h: int,
    n_bins: int = 80,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute a 2-D histogram of gaze density in screen coordinates."""
    # Starter scaffold — fill in the ... blanks, then uncomment:
    # finite = np.isfinite(xs) & np.isfinite(...)
    # return np.histogram2d(
    #     xs[...], ys[...],
    #     bins=...,
    #     range=[[0, screen_w], [0, ...]],
    # )
    raise NotImplementedError("Complete build_gaze_heatmap")

In [ ]:
def test_build_gaze_heatmap(fn):
    xs_t = np.full(100, 50.0)
    ys_t = np.full(100, 50.0)
    hm, xe, ye = fn(xs_t, ys_t, screen_w=1000, screen_h=1000, n_bins=10)

    assert isinstance(hm, np.ndarray), "First return value must be a numpy array."
    assert hm.shape == (10, 10), f"Shape must be (10, 10), got {hm.shape}."
    assert len(xe) == 11 and len(ye) == 11, "Edges must have n_bins + 1 elements."
    assert hm[0, 0] == 100, f"All points should fall in bin [0, 0], got {hm[0, 0]}."
    assert hm.sum() == 100, "Total bin count must equal the number of input points."

    print("Exercise 7 passed.")


test_build_gaze_heatmap(build_gaze_heatmap)


---

## Configuration

Choose a participant and set the screen/marker parameters. `DETECT_EVERY_N = 5` keeps the live workshop pass faster; lower it if you need denser AprilTag checks.


In [ ]:
EXPERIMENT             = "IMRFSpatialAV"
PARTICIPANT_ID         = DEFAULT_PARTICIPANT_ID  # options: "p0096", "p0097", "p0099"
PREFERRED_RECORDING_ID = None

SCREEN_W              = 1920
SCREEN_H              = 1080
MONITOR_WIDTH_CM      = 52.0
VIEWING_DISTANCE_CM   = 60.0

TAG_FAMILY            = "tag36h11"
DETECT_EVERY_N        = 5

CLEAN_METHOD          = "linear"
BLINK_PAD_MS          = 50.0
LOWPASS_HZ            = 30.0
HEATMAP_BINS          = 80

NEON_ROOT = PROJECT_ROOT / "data_output" / EXPERIMENT / "neon"
RECORDING_PATH = find_neon_recording(
    NEON_ROOT,
    participant_id=PARTICIPANT_ID,
    recording_id=PREFERRED_RECORDING_ID,
)
if RECORDING_PATH is None or not RECORDING_PATH.is_dir():
    raise FileNotFoundError(f"No Neon recording found under {NEON_ROOT} for {PARTICIPANT_ID!r}")

SELECTED_PARTICIPANT = neon_participant_id(RECORDING_PATH) or PARTICIPANT_ID
PARTICIPANT = RECORDING_PATH.name
OUTPUT_DIR  = PROJECT_ROOT / "notebooks" / "workshop_output" / EXPERIMENT / PARTICIPANT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Participant: {SELECTED_PARTICIPANT}")
print(f"Recording : {PARTICIPANT}")
print(f"Screen    : {SCREEN_W} x {SCREEN_H} px")
print(f"Output    : {OUTPUT_DIR}")


## Load the Neon Recording

Open the recording, rename gaze columns for the cleaning helper, and build the scene-frame timestamp array.


In [ ]:
try:
    import pupil_labs.neon_recording as nr
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Select the imrf_env_notebook kernel or install pupil-labs-neon-recording."
    ) from exc

recording = nr.open(str(RECORDING_PATH))

gaze_df = recording.gaze.pd.copy().rename(columns={
    "time": "timestamp_ns",
    "point_x": "gaze_x_scene",
    "point_y": "gaze_y_scene",
})
if gaze_df["timestamp_ns"].abs().max() < 1e12:
    gaze_df["timestamp_ns"] = (gaze_df["timestamp_ns"] * 1e9).astype(np.int64)

fix_df = recording.fixations.pd.copy() if recording.fixations is not None else pd.DataFrame()

scene_t = np.asarray(recording.scene.time, dtype=float)
if scene_t[np.isfinite(scene_t)].max() < 1e12:
    scene_t = scene_t * 1e9
scene_times_ns = scene_t.astype(np.int64)
n_frames = len(scene_times_ns)

print(f"Gaze samples : {len(gaze_df):,}")
print(f"Fixations    : {len(fix_df):,}")
print(f"Scene frames : {n_frames:,}")


## Gaze Cleaning

Use the same blink interpolation and lowpass step as the interpolation notebook.


In [ ]:
gaze_df = clean_gaze_stream(
    gaze_df,
    recording,
    method=CLEAN_METHOD,
    blink_pad_ms=BLINK_PAD_MS,
    lowpass_hz=LOWPASS_HZ,
)
print("Gaze cleaning complete.")

## AprilTag Detection and Homography Estimation

Detect the border AprilTags on every N-th frame, estimate fresh homographies, then forward-fill the latest valid homography for frames between detections.


In [ ]:
margin_px = compute_marker_margin_px(
    screen_width_px=SCREEN_W,
    monitor_width_cm=MONITOR_WIDTH_CM,
    viewing_distance_cm=VIEWING_DISTANCE_CM,
)
w, h, m = SCREEN_W, SCREEN_H, margin_px
MARKER_POSITIONS = {
    10: (m,     h - m),  11: (m,     h / 2),
    12: (m,     m    ),  13: (w / 2, m    ),
    14: (w - m, m    ),  15: (w - m, h / 2),
    16: (w - m, h - m),  17: (w / 2, h - m),
}
mapper = ScreenMapper(ScreenMapperConfig(
    screen_size=(SCREEN_W, SCREEN_H),
    marker_positions=MARKER_POSITIONS,
    tag_family=TAG_FAMILY,
    min_markers=4,
    on_screen_margin_px=0.0,
))


def _gray_frame(fidx: int) -> np.ndarray | None:
    try:
        frame = recording.scene.sample(np.array([int(scene_times_ns[fidx])]))[0]
        if hasattr(frame, "gray"):
            return np.asarray(frame.gray, dtype=np.uint8)
        if hasattr(frame, "bgr"):
            return cv2.cvtColor(np.asarray(frame.bgr, dtype=np.uint8), cv2.COLOR_BGR2GRAY)
    except Exception:
        return None
    return None


raw_H = {}
total = (n_frames + DETECT_EVERY_N - 1) // DETECT_EVERY_N
for step, fidx in enumerate(range(0, n_frames, DETECT_EVERY_N), start=1):
    gray = _gray_frame(fidx)
    raw_H[fidx] = mapper.process_frame(gray, use_cache=False) if gray is not None else None
    if step % 100 == 0 or step == total:
        print(f"  {step}/{total} sampled frames", end="\r")

all_H = {}
last_valid = None
for fidx in range(n_frames):
    if raw_H.get(fidx) is not None:
        last_valid = raw_H[fidx]
    all_H[fidx] = last_valid

fresh_cov = sum(H is not None for H in raw_H.values()) / max(len(raw_H), 1)
print(f"\nDetection done. Fresh coverage on sampled frames: {fresh_cov:.1%}")


## Map Gaze to Screen Coordinates

Exercise 6 assigns each gaze sample to a frame; the helper below then applies that frame's homography in batches.


In [ ]:
def map_points_by_frame(points_xy: np.ndarray, frame_indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    screen_xy = np.full((len(points_xy), 2), np.nan)
    on_screen = np.zeros(len(points_xy), dtype=bool)

    for fidx in np.unique(frame_indices):
        H = all_H.get(int(fidx))
        if H is None:
            continue
        rows = np.flatnonzero(frame_indices == fidx)
        pts = points_xy[rows]
        valid = np.isfinite(pts).all(axis=1)
        if not valid.any():
            continue
        mapped = mapper.map_gaze(pts[valid], H)
        screen_xy[rows[valid]] = mapped
        on_screen[rows[valid]] = mapper.is_on_screen(mapped)

    return screen_xy, on_screen


gaze_ts_ns = gaze_df["timestamp_ns"].to_numpy(dtype=np.int64)
frame_indices = assign_gaze_to_frames(gaze_ts_ns, scene_times_ns)
scene_xy = gaze_df[["gaze_x_scene", "gaze_y_scene"]].to_numpy(dtype=float)

screen_xy, on_screen = map_points_by_frame(scene_xy, frame_indices)
gaze_df["gaze_x_screen"] = screen_xy[:, 0]
gaze_df["gaze_y_screen"] = screen_xy[:, 1]
gaze_df["on_screen"] = on_screen
gaze_df["frame_idx"] = frame_indices

n_on = int(on_screen.sum())
print(f"Gaze mapped: {n_on:,} / {len(gaze_df):,} on screen ({100 * n_on / len(gaze_df):.1f}%)")


## Map Fixations to Screen Coordinates

Fixation centroids use the same timestamp-to-frame assignment and batched homography mapping.


In [ ]:
_FIX_X = next((c for c in ["mean_gaze_x", "start_gaze_x", "x"] if c in fix_df.columns), None)
_FIX_Y = next((c for c in ["mean_gaze_y", "start_gaze_y", "y"] if c in fix_df.columns), None)
_FIX_T = next((c for c in ["timestamp_ns", "start_time", "timestamp"] if c in fix_df.columns), None)

if fix_df.empty or _FIX_X is None or _FIX_Y is None or _FIX_T is None:
    print("No fixation data or missing centroid/timestamp columns.")
else:
    fix_ts_ns = fix_df[_FIX_T].to_numpy(dtype=np.int64)
    if fix_ts_ns.max() < 1e12:
        fix_ts_ns = (fix_ts_ns * 1e9).astype(np.int64)

    fix_fidxs = assign_gaze_to_frames(fix_ts_ns, scene_times_ns)
    fix_xy = fix_df[[_FIX_X, _FIX_Y]].to_numpy(dtype=float)
    fix_screen_xy, fix_on = map_points_by_frame(fix_xy, fix_fidxs)

    fix_df["screen_x"] = fix_screen_xy[:, 0]
    fix_df["screen_y"] = fix_screen_xy[:, 1]
    fix_df["on_screen"] = fix_on
    fix_df["frame_idx"] = fix_fidxs
    fix_df["event_x"] = fix_df[_FIX_X]
    fix_df["event_y"] = fix_df[_FIX_Y]

    if "duration" in fix_df.columns:
        dur = fix_df["duration"].to_numpy(dtype=float)
        fix_df["duration_s"] = dur / 1e9 if np.nanmedian(dur) > 1e6 else dur
    elif "stop_time" in fix_df.columns:
        dur = fix_df["stop_time"].to_numpy(dtype=float) - fix_df[_FIX_T].to_numpy(dtype=float)
        fix_df["duration_s"] = dur / 1e9 if np.nanmedian(np.abs(dur)) > 1e6 else dur
    else:
        fix_df["duration_s"] = 0.3

    n_on_fix = int(fix_on.sum())
    print(f"Fixations mapped: {n_on_fix:,} / {len(fix_df):,} on screen ({100 * n_on_fix / len(fix_df):.1f}%)")


## Gaze Heatmap

Apply Exercise 7 to the on-screen gaze samples.


In [ ]:
on_gaze = gaze_df[gaze_df["on_screen"]]
xs_on = on_gaze["gaze_x_screen"].to_numpy(dtype=float)
ys_on = on_gaze["gaze_y_screen"].to_numpy(dtype=float)

hm, xe, ye = build_gaze_heatmap(xs_on, ys_on, SCREEN_W, SCREEN_H, n_bins=HEATMAP_BINS)

fig, ax = plt.subplots(figsize=(12, 12 * SCREEN_H / SCREEN_W), constrained_layout=True)
im = ax.imshow(
    hm.T,
    extent=[xe[0], xe[-1], ye[-1], ye[0]],
    origin="upper",
    cmap=PLOT_COLORS["cmap_heatmap"],
    aspect="auto",
    interpolation="gaussian",
)
ax.set_xlim(0, SCREEN_W)
ax.set_ylim(SCREEN_H, 0)
ax.set_xlabel("Screen X (px)")
ax.set_ylabel("Screen Y (px)")
ax.set_title(f"Gaze Heatmap — {PARTICIPANT} ({len(xs_on):,} on-screen samples)")
fig.colorbar(im, ax=ax, label="Gaze sample count")
plt.show()


## Fixation Scanpath

A quick screen-space view of fixation order and duration.


In [ ]:
if fix_df.empty or "screen_x" not in fix_df.columns:
    print("No mapped fixation data to plot.")
else:
    on_fix = fix_df[fix_df["on_screen"]].sort_values(_FIX_T).reset_index(drop=True)
    if on_fix.empty:
        print("No on-screen fixations to plot.")
    else:
        order = np.arange(len(on_fix))
        sizes = np.clip(on_fix.get("duration_s", pd.Series(0.3, index=on_fix.index)) * 800, 50, 500)

        fig, ax = plt.subplots(figsize=(12, 12 * SCREEN_H / SCREEN_W), constrained_layout=True)
        ax.plot(on_fix["screen_x"], on_fix["screen_y"], color=PLOT_COLORS["saccade"], lw=1.0, alpha=0.45)
        sc = ax.scatter(
            on_fix["screen_x"], on_fix["screen_y"],
            c=order, s=sizes, cmap=PLOT_COLORS.get("cmap_fixations", "viridis"),
            alpha=0.85, edgecolors=PLOT_COLORS["zero_line"], linewidths=0.6,
        )
        ax.set_xlim(0, SCREEN_W)
        ax.set_ylim(SCREEN_H, 0)
        ax.set_aspect("equal")
        ax.set_xlabel("Screen X (px)")
        ax.set_ylabel("Screen Y (px)")
        ax.set_title(f"Fixation Scanpath — {PARTICIPANT} ({len(on_fix)} on-screen fixations)")
        fig.colorbar(sc, ax=ax, label="Fixation order")
        plt.show()


## Export Results

Save the three CSVs used by notebook 03. The optional overlay video is off by default because rendering it can take a few minutes.


In [ ]:
RENDER_VIDEO = False

gaze_df.to_csv(OUTPUT_DIR / "apriltag_mapped_gaze.csv", index=False)
fix_df.to_csv(OUTPUT_DIR / "apriltag_mapped_fixations.csv", index=False)

if not fix_df.empty and "screen_x" in fix_df.columns and _FIX_T is not None:
    scanpath = fix_df[fix_df["on_screen"]].sort_values(_FIX_T).reset_index(drop=True)
else:
    scanpath = fix_df.iloc[0:0].copy()
scanpath.to_csv(OUTPUT_DIR / "apriltag_scanpath.csv", index=False)

print(f"CSVs saved to {OUTPUT_DIR}")

if RENDER_VIDEO:
    from libs.analysis.screen_gaze_visualizer import render_scene_overlay_video
    video_path = OUTPUT_DIR / "scene_overlay.mp4"
    render_scene_overlay_video(
        recording_path=RECORDING_PATH,
        gaze_df=gaze_df,
        fixations_df=fix_df,
        scanpath_df=scanpath,
        output_path=video_path,
    )
    print(f"Video saved to {video_path}")
else:
    print("Video skipped. Set RENDER_VIDEO = True to render scene_overlay.mp4.")


## Short Reflection

Answer these in a markdown cell below, or discuss as a group:

1. Why does a projective homography need at least 4 corresponding points?
2. What does low fresh AprilTag coverage imply for the reliability of the mapped gaze?
3. Does the heatmap concentrate on expected screen regions?
4. Do early and late fixations cluster differently in the scanpath?
